In [ ]:
from openai import OpenAI

VLLM_API_BASE = "http://127.0.0.1:8000/v1"
MODEL_NAME = "qwen3-8b"
_client = OpenAI(base_url=VLLM_API_BASE, api_key="EMPTY")

def chat(messages, *, enable_thinking=False, temperature=0.7, top_p=0.95,
         top_k=20, max_tokens=2048):
    if enable_thinking and temperature > 0.6:
        temperature = 0.6
    resp = _client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
        extra_body={
            "top_k": top_k,
            "chat_template_kwargs": {"enable_thinking": enable_thinking},
        },
    )
    msg = resp.choices[0].message
    return {
        "content": (msg.content or "").strip(),
        "reasoning": getattr(msg, "reasoning_content", None),
        "finish_reason": resp.choices[0].finish_reason,
        "usage": dict(resp.usage),
    }

In [ ]:
# Test 1 : non-thinking
r = chat(
    [{"role": "user", "content": "What is perplexity in NLP? Two sentences max."}],
    enable_thinking=False,
    max_tokens=256,
)
print("=== NON-THINKING ===")
print(r["content"])
print(f"tokens: {r['usage']}")
print()

# Test 2 : thinking
r = chat(
    [{"role": "user", "content": "Is 9.11 greater than 9.8?"}],
    enable_thinking=True,
    max_tokens=2048,
)
print("=== THINKING ===")
print(f"reasoning ({len(r['reasoning'])} chars): {r['reasoning'][:200]}...")
print(f"content: {r['content']}")
print(f"tokens: {r['usage']}")

In [ ]:
import random
label = [random.randint(0, 1) for _ in range(50)]
target = [random.uniform(0, 1) for _ in range(50)]

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# label : ground truth binaire (0/1) ; target : score continu du détecteur
fpr, tpr, thresholds = roc_curve(label, target)
roc_auc = auc(fpr, tpr)

# Seuil maximisant TPR sous contrainte FPR <= 5%
mask = fpr <= 0.05
idx = np.argmax(tpr[mask])                 # meilleur TPR dans la zone admissible
thr_at_5fpr = thresholds[mask][idx]
tpr_at_5fpr = tpr[mask][idx]
fpr_at_5fpr = fpr[mask][idx]

plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, label=f"ROC (AUC = {roc_auc:.4f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.4)
plt.axvline(0.05, color="red", linestyle=":", label="FPR = 5%")
plt.scatter([fpr_at_5fpr], [tpr_at_5fpr], color="red", zorder=5,
            label=f"TPR@5%FPR = {tpr_at_5fpr:.3f} (thr={thr_at_5fpr:.4f})")
plt.xlabel("FPR"); plt.ylabel("TPR")
plt.title("ROC curve"); plt.legend(loc="lower right")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print(f"AUROC          : {roc_auc:.4f}")
print(f"TPR @ 5% FPR   : {tpr_at_5fpr:.4f}")
print(f"Threshold      : {thr_at_5fpr:.6f}")